# Setup and Testing Model's Innate Knowledge 

Load variables from the .env file

In [21]:
import sys
sys.path.insert(0, "..")
from config import API_KEY as key, ENDPOINT_BASE as endpoint_base


## Validate that configration is correct

In [22]:
print("CONFIG DETAILS")
print("=====================")
print(f"API Key = {key}")
print(f"EndPoint Base URL = {endpoint_base}")


CONFIG DETAILS
API Key = sk-UFHcLOTZk_73o6YwVQhSiQ
EndPoint Base URL = https://litellm-prod.apps.maas.redhatworkshops.io/v1


In [23]:
import requests
url_models = f"{endpoint_base}/models"
url_chat = f"{endpoint_base}/chat/completions"

## Test the Connection to MaaS Server

The following code will test the connection to the MaaS endpoint and ensure you can communicate with the LLMs hosted there.

In [24]:
headers = {
    "Authorization": f"Bearer {key}",
    "Content-Type": "application/json"
}
data = {
    "model": "granite-3-2-8b-instruct",
    "messages": [{"role": "user", "content": "Hello, world!"}]
}

response = requests.post(url_chat, headers=headers, json=data)
print(response.json())

{'id': 'chatcmpl-cfa8c79acbf4438f897ee795725b591f', 'created': 1770919117, 'model': 'granite-3-2-8b-instruct', 'object': 'chat.completion', 'system_fingerprint': None, 'choices': [{'finish_reason': 'stop', 'index': 0, 'message': {'content': 'Hello! How can I assist you today?', 'role': 'assistant', 'tool_calls': None, 'function_call': None}, 'provider_specific_fields': {'stop_reason': None}}], 'usage': {'completion_tokens': 10, 'prompt_tokens': 12, 'total_tokens': 22, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'service_tier': None, 'prompt_logprobs': None, 'kv_transfer_params': None}


## Check the models available to the API Key

In [25]:
headers = {"Authorization": f"Bearer {key}"}

response = requests.get(url_models, headers=headers)
models = response.json()

print("Models available")

# Loop through and print just the names
for model in models['data']:
    print(f"\t- {model['id']}")

Models available
	- qwen3-14b
	- granite-4-0-h-tiny
	- granite-3-2-8b-instruct
	- microsoft-phi-4


## Ask a common RPG question

### Import and Variable

In [26]:
import requests

headers = {"Authorization": f"Bearer {key}"}
user_question = "Why can’t Elves roll higher than a d6 for hit points?"

### Ask all the models the same question

In [27]:
# 2. Get the list of all models
response = requests.get(url_models, headers=headers)
models_data = response.json().get('data', [])

print(f"--- Running prompt against {len(models_data)} models ---\n")

# 3. Loop through each model and ask the question
for model in models_data:
    model_id = model['id']
    print(f"🤖 Testing Model: {model_id}...")
    
    payload = {
        "model": model_id,
        "messages": [{"role": "user", "content": user_question}],
        "max_tokens": 100  # Keeping it short for the test
    }
    
    try:
        res = requests.post(url_chat, headers=headers, json=payload)
        res.raise_for_status() # Check for HTTP errors
        
        answer = res.json()['choices'][0]['message']['content']
        print(f"✅ Response:\n{answer}\n")
        
    except Exception as e:
        print(f"❌ Failed to get response from {model_id}: {e}\n")

print("--- All tests complete ---")

--- Running prompt against 4 models ---

🤖 Testing Model: qwen3-14b...
✅ Response:
<think>
Okay, so the user is asking why Elves can't roll higher than a d6 for hit points. Hmm, I need to figure out what they're referring to. I know that in D&D 5e, different races have different starting hit points. For example, Elves start with 1d6 + Constitution modifier for hit points, while other races might have different dice. But why can't Elves roll higher than a d6? Maybe they're confused because some races use a

🤖 Testing Model: granite-4-0-h-tiny...
✅ Response:
The concept of Elves rolling higher than a d6 for hit points is not universally applicable across all fantasy role-playing games. The rules can vary greatly depending on the specific game or edition being referenced. 

However, to provide some general context: 

In many role-playing games, different races and classes often have unique abilities, strengths, and limitations. This can influence how hit points are calculated or modified.

### Define a function

In [28]:
def test_all_models(api_key: str, base_url: str, question: str, max_tokens: int = 1000):

    url_models = f"{base_url}/models"
    url_chat = f"{base_url}/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    # 1. Get all models
    try:
        response = requests.get(url_models, headers=headers)
        response.raise_for_status()
        models_data = response.json().get("data", [])
    except Exception as e:
        print(f"❌ Failed to fetch models: {e}")
        return

    print(f"\n--- Running prompt against {len(models_data)} models ---\n")

    # 2. Loop through each model
    for model in models_data:
        model_id = model["id"]
        print(f"🤖 Testing Model: {model_id}...")

        payload = {
            "model": model_id,
            "messages": [{"role": "user", "content": question}],
            "max_tokens": max_tokens
        }

        try:
            res = requests.post(url_chat, headers=headers, json=payload)
            res.raise_for_status()

            answer = res.json()["choices"][0]["message"]["content"]
            print(f"✅ Response:\n{answer}\n")

        except Exception as e:
            print(f"❌ Failed to get response from {model_id}: {e}\n")

    print("--- All tests complete ---")

### Ask a question
This question is specific to our customer's rulebook and reveals a knowledge gap that the models have.

In [29]:
test_all_models(
    api_key=key,
    base_url=endpoint_base,
    question="What happens if a Thief fails an Open Locks attempt?"
)


--- Running prompt against 4 models ---

🤖 Testing Model: qwen3-14b...
✅ Response:


In **Dungeons & Dragons 5th Edition**, if a Thief (or any character using **Thieves' Tools** to attempt **Open Locks**) fails a **Dexterity check**, the **lock remains unopened**, and the attempt is unsuccessful. Here's a breakdown of the standard rules and possible consequences:

### **Standard Outcome:**
- **Failed Check:** The lock is not opened. The character must try again, potentially with a new roll or under different circumstances (e.g., using a different tool, gaining advantage, or spending time to study the lock).

### **Optional Consequences (DM Discretion):**
While the core rule is straightforward, a **Dungeon Master (DM)** may impose additional effects depending on the situation:
1. **Lock Damage:** A failed attempt might damage the lock, making it harder to open in the future or requiring a **Disarm Trap** check to repair it.
2. **Triggering a Trap:** If the lock is part of a **trapped m